# Combine NQM Flat Files

Use this notebook to combine multiple monthly NQM pool flat files into one tab-delimited flat file with a single header row.

The default settings combine the ten pools from the ENU top-10 request into `NQM_TOP10_20260401.txt` under `N:/FlatFilesMonthly/NQMMonthly/`.

In [13]:
from pathlib import Path

# Edit these parameters as needed.
source_dir = Path(r"N:/FlatFilesMonthly/NQMMonthly")
asof_date = "20260501"
output_pool_id = "NQM_TOP10"
output_path = source_dir / f"{output_pool_id}_{asof_date}.txt"

pool_ids = [
    "B3A",  # VERUS 2022-1
    "ENU",  # VISIO 2023-1
    "ESU",  # CHNGE 2023-4
    "B4O",  # COLT 2022-3
    "B4U",  # VERUS 2022-3
    "ES8",  # ADMT 2023-NQM3
    "EVD",  # ADMT 2024-NQM1
    "B9U",  # CHNGE 2022-1
    "B9V",  # CHNGE 2022-2
    "Q4Y",  # BRAVO 2021-NQM1
]

source_files = [source_dir / f"{pool_id}_{asof_date}.txt" for pool_id in pool_ids]
source_files

[WindowsPath('N:/FlatFilesMonthly/NQMMonthly/B3A_20260501.txt'),
 WindowsPath('N:/FlatFilesMonthly/NQMMonthly/ENU_20260501.txt'),
 WindowsPath('N:/FlatFilesMonthly/NQMMonthly/ESU_20260501.txt'),
 WindowsPath('N:/FlatFilesMonthly/NQMMonthly/B4O_20260501.txt'),
 WindowsPath('N:/FlatFilesMonthly/NQMMonthly/B4U_20260501.txt'),
 WindowsPath('N:/FlatFilesMonthly/NQMMonthly/ES8_20260501.txt'),
 WindowsPath('N:/FlatFilesMonthly/NQMMonthly/EVD_20260501.txt'),
 WindowsPath('N:/FlatFilesMonthly/NQMMonthly/B9U_20260501.txt'),
 WindowsPath('N:/FlatFilesMonthly/NQMMonthly/B9V_20260501.txt'),
 WindowsPath('N:/FlatFilesMonthly/NQMMonthly/Q4Y_20260501.txt')]

## List All NQM Pool IDs and Deal Names

This section scans `N:/FlatFilesMonthly/NQMMonthly` for actual monthly pool flat files, excludes generated pseudo files like `NQMAGE*` and `NQMDOCTYPE*`, extracts unique pool IDs from filenames, and looks up deal names from `libremax.dbo.CRTStats`.

In [14]:
import pandas as pd
import lmdata.lmdb as lmdb

pseudo_file_prefixes = (
    "NQM_TOP",
    "NQM_ALL",
    "NQMAGE",
    "NQMDOCTYPE",
    "NQMFICO",
    "NQMPRODTYPE",
    "NQMUPDLTV",
    "NQMWAC",
)

def get_pool_id_from_flatfile(path: Path) -> str | None:
    pool_id, file_asof_date = path.stem.rsplit("_", 1)
    if file_asof_date != asof_date:
        return None
    if pool_id.startswith(pseudo_file_prefixes):
        return None
    return pool_id

all_nqm_files = sorted(source_dir.glob("*.txt"))
actual_pool_files = [path for path in all_nqm_files if get_pool_id_from_flatfile(path) is not None]
pool_file_lookup = {get_pool_id_from_flatfile(path): path for path in actual_pool_files}
all_pool_ids = sorted(pool_file_lookup)

pool_id_list = ", ".join(f"'{pool_id}'" for pool_id in all_pool_ids)

sql_string = f"""
    select distinct
        pool_id,
        bbg_deal_name
    from libremax.dbo.CRTStats
    where pool_id in ({pool_id_list})
    order by pool_id
"""

pool_deal_lookup = lmdb.executeSQLTableDF(db="libremax", sql_query=sql_string, redshift=False)

pool_id_df = pd.DataFrame({"pool_id": all_pool_ids})
pool_deal_lookup = pool_id_df.merge(pool_deal_lookup, on="pool_id", how="left")

print(f"Found {len(all_pool_ids):,} unique pool IDs from {len(actual_pool_files):,} actual NQM monthly flat files")
print(f"Ignored {len(all_nqm_files) - len(actual_pool_files):,} pseudo or off-date files")
display(pool_deal_lookup)

len(actual_pool_files), len(all_pool_ids)

Found 593 unique pool IDs from 593 actual NQM monthly flat files
Ignored 739 pseudo or off-date files


,pool_id,bbg_deal_name
0,8HQ,IMPRL 2022-NQM4
1,B30,CSMC 2022-NQM1
2,B31,IMPRL 2020-NQM1
3,B32,IMPRL 2021-NQM1
4,B33,COLT 2021-6
...,...,...
588,QZP,NRZT 2021-NQ1R
589,QZQ,GCAT 2021-NQM1
590,QZT,DRMT 2021-1
591,QZU,CSMC 2021-NQM2


(593, 593)

## Select Up to 100 Deals

Pick a reproducible subset by vintage and deal shelf. The selection favors larger files within each shelf, but rotates across shelves so one issuer does not dominate the combined flat file.

In [15]:
import re

vintage_targets = {
    "2019_and_earlier": 5,
    "2020": 6,
    "2021": 10,
    "2022": 14,
    "2023": 18,
    "2024": 22,
    "2025": 20,
    "2026": 5,
}

def count_data_rows(path: Path) -> int:
    with path.open("rb") as f:
        return sum(1 for _ in f) - 1

def extract_vintage(deal_name: str) -> str | None:
    match = re.search(r"20\d{2}", deal_name)
    return match.group(0) if match else None

def vintage_bucket(vintage: str) -> str:
    if int(vintage) <= 2019:
        return "2019_and_earlier"
    return vintage

def pick_diverse_deals(vintage_df: pd.DataFrame, target_count: int) -> pd.DataFrame:
    ranked = vintage_df.sort_values(
        ["shelf", "data_rows", "bbg_deal_name"],
        ascending=[True, False, True],
    ).copy()
    ranked["shelf_rank"] = ranked.groupby("shelf").cumcount()
    return ranked.sort_values(
        ["shelf_rank", "data_rows", "shelf", "bbg_deal_name"],
        ascending=[True, False, True, True],
    ).head(target_count)

deal_selection_df = pool_deal_lookup.dropna(subset=["bbg_deal_name"]).copy()
deal_selection_df["file_path"] = deal_selection_df["pool_id"].map(pool_file_lookup)
deal_selection_df["data_rows"] = deal_selection_df["file_path"].map(count_data_rows)
deal_selection_df["shelf"] = deal_selection_df["bbg_deal_name"].str.split().str[0]
deal_selection_df["vintage"] = deal_selection_df["bbg_deal_name"].map(extract_vintage)
deal_selection_df = deal_selection_df.dropna(subset=["vintage"])
deal_selection_df["vintage_bucket"] = deal_selection_df["vintage"].map(vintage_bucket)

selected_deals = pd.concat(
    [
        pick_diverse_deals(
            deal_selection_df[deal_selection_df["vintage_bucket"] == vintage],
            target_count,
        )
        for vintage, target_count in vintage_targets.items()
    ],
    ignore_index=True,
).sort_values(["vintage_bucket", "shelf", "bbg_deal_name"])

source_files = selected_deals["file_path"].tolist()
output_pool_id = "NQM_SELECTED100"
output_path = source_dir / f"{output_pool_id}_{asof_date}.txt"

print(f"Selected {len(selected_deals):,} deals with {selected_deals['data_rows'].sum():,} total expected rows")
display(selected_deals.groupby("vintage_bucket").agg(deals=("pool_id", "count"), rows=("data_rows", "sum")))
display(pd.crosstab(selected_deals["vintage_bucket"], selected_deals["shelf"]))
display(selected_deals[["pool_id", "bbg_deal_name", "vintage_bucket", "shelf", "data_rows"]])

len(source_files), output_path

Selected 100 deals with 92,849 total expected rows


,deals,rows
vintage_bucket,,
2019_and_earlier,5,5849
2020,6,3866
2021,10,6619
2022,14,12086
2023,18,12097
2024,22,22193
2025,20,23115
2026,5,7024


shelf,ACRA,ADMT,AOMT,ARRW,BARC,BINOM,BRAVO,CIM,COLT,CROSS,...,NLT,NRZT,NYMT,OBX,PRKCM,PRPM,STAR,TRK,VERUS,VISIO
vintage_bucket,,,,,,,,,,,,,,,,,,,,,
2019_and_earlier,0,0,1,1,0,0,1,0,0,0,...,0,1,0,0,0,0,0,0,0,0
2020,0,0,1,0,0,0,0,0,0,0,...,0,1,0,0,0,0,1,0,0,0
2021,0,0,1,0,0,1,0,0,0,0,...,1,0,0,1,1,0,0,0,1,0
2022,0,0,1,1,1,1,0,0,0,0,...,0,0,1,0,0,1,0,1,1,0
2023,0,1,1,0,1,0,1,1,0,0,...,0,1,0,1,1,1,0,0,1,1
2024,1,1,1,0,1,0,1,0,1,1,...,0,1,1,1,1,1,0,0,1,0
2025,0,1,0,0,1,0,1,1,0,1,...,1,1,1,1,1,1,0,0,1,0
2026,0,0,0,0,0,0,0,0,0,0,...,0,0,1,1,0,0,0,0,1,0


,pool_id,bbg_deal_name,vintage_bucket,shelf,data_rows
3,QC2,AOMT 2019-6,2019_and_earlier,AOMT,372
2,QVX,ARRW 2019-2,2019_and_earlier,ARRW,454
1,QC6,BRAVO 2019-2,2019_and_earlier,BRAVO,1392
4,QQS,FSMT 2018-3INV,2019_and_earlier,FSMT,293
0,QP6,NRZT 2018-1A,2019_and_earlier,NRZT,3338
...,...,...,...,...,...
99,HWI,GSMBS 2026-NQM1,2026,GSMBS,1019
98,HWP,JPMMT 2026-NQM1,2026,JPMMT,1162
95,HW3,NYMT 2026-INV1,2026,NYMT,1713
96,HXF,OBX 2026-NQM1,2026,OBX,1599


(100,
 WindowsPath('N:/FlatFilesMonthly/NQMMonthly/NQM_SELECTED100_20260501.txt'))

## Select Deals from Explicit Deal Name Filter

Use this section when you want the combined flat file to use a hand-selected deal list instead of the stratified 100-deal sample above. Run this cell after the CRT lookup cell and before the count/combine cells.

python agencydata\wh_lp_update.py --deal_downloaddate_end 20260301 --deal_downloaddate_start 20260301 --force_save_trans --force_skip_trans --deal_list "AOMT 2019-6,ARRW 2019-2,BRAVO 2019-2,FSMT 2018-3INV,NRZT 2018-1A,AOMT 2020-3,CSMC 2020-SPT1,MFRA 2020-NQM2,NRZT 2020-2A,OBX 2020-EXP2,STAR 2020-3,AOMT 2021-6,BINOM 2021-INV1,CSMC 2021-NQM8,DRMT 2021-4,IMPRL 2021-NQM3,MFRA 2021-INV2,NLT 2021-INV2,OBX 2021-NQM4,PRKCM 2021-AFC2,VERUS 2021-7,AOMT 2022-1,ARRW 2022-2,BARC 2022-INV1,BINOM 2022-INV1,CSMC 2022-NQM3,EFMT 2022-2,GCAT 2022-NQM3,IMPRL 2022-NQM1,JPMMT 2022-DSC1,MFRA 2022-INV1,NYMT 2022-INV1,PRPM 2022-INV1,TRK 2022-INV1,VERUS 2022-2,ADMT 2023-NQM4,AOMT 2023-1,BRAVO 2023-NQM1,CIM 2023-I2,EFMT 2023-1,GCAT 2023-NQM3,GSMBS 2023-NQM1,HOMES 2023-NQM1,IMPRL 2023-NQM1,JPMMT 2023-DSC1,MCMLT 2023-NQM1,MFRA 2023-INV2,MSRM 2023-NQM1,NRZT 2023-NQM1,OBX 2023-NQM8,PRKCM 2023-AFC1,VERUS 2023-7,VISIO 2023-2,ACRA 2024-NQM1,ADMT 2024-NQM5,AOMT 2024-6,BARC 2024-NQM2,BRAVO 2024-NQM5,COLT 2024-INV2,CROSS 2024-H7,DRMT 2024-1,EFMT 2024-INV2,GRADE 2024-INV1,GSMBS 2024-NQM1,HOMES 2024-AFC1,JPMMT 2024-VIS1,LMAT 2024-INV1,MFRA 2024-NQM3,MSRM 2024-NQM4,NRZT 2024-NQM2,NYMT 2024-INV1,OBX 2024-NQM15,PRKCM 2024-AFC1,PRPM 2024-NQM1,VERUS 2024-9,ADMT 2025-NQM4,BARC 2025-NQM3,BRAVO 2025-NQM8,CIM 2025-I1,CROSS 2025-H7,DRMT 2025-INV1,EFMT 2025-INV5,GCAT 2025-NQM3,GSMBS 2025-DSC2,HOMES 2025-AFC4,JPMMT 2025-NQM4,MFRA 2025-NQM5,MSRM 2025-DSC2,NLT 2025-INV1,NRZT 2025-NQM6,NYMT 2025-INV1,OBX 2025-NQM21,PRKCM 2025-AFC2,PRPM 2025-NQM1,VERUS 2025-11,GSMBS 2026-NQM1,JPMMT 2026-NQM1,NYMT 2026-INV1,OBX 2026-NQM1,VERUS 2026-1"

In [16]:
import warnings

deal_name_filter = [
    "AOMT 2019-6", "ARRW 2019-2", "BRAVO 2019-2", "FSMT 2018-3INV",
    "NRZT 2018-1A", "AOMT 2020-3", "CSMC 2020-SPT1", "MFRA 2020-NQM2",
    "NRZT 2020-2A", "OBX 2020-EXP2", "STAR 2020-3", "AOMT 2021-6",
    "BINOM 2021-INV1", "CSMC 2021-NQM8", "DRMT 2021-4",
    "IMPRL 2021-NQM3", "MFRA 2021-INV2", "NLT 2021-INV2",
    "OBX 2021-NQM4", "PRKCM 2021-AFC2", "VERUS 2021-7", "AOMT 2022-1",
    "ARRW 2022-2", "BARC 2022-INV1", "BINOM 2022-INV1",
    "CSMC 2022-NQM3", "EFMT 2022-2", "GCAT 2022-NQM3",
    "IMPRL 2022-NQM1", "JPMMT 2022-DSC1", "MFRA 2022-INV1",
    "NYMT 2022-INV1", "PRPM 2022-INV1", "TRK 2022-INV1",
    "VERUS 2022-2", "ADMT 2023-NQM4", "AOMT 2023-1", "BRAVO 2023-NQM1",
    "CIM 2023-I2", "EFMT 2023-1", "GCAT 2023-NQM3", "GSMBS 2023-NQM1",
    "HOMES 2023-NQM1", "IMPRL 2023-NQM1", "JPMMT 2023-DSC1",
    "MCMLT 2023-NQM1", "MFRA 2023-INV2", "MSRM 2023-NQM1",
    "NRZT 2023-NQM1", "OBX 2023-NQM8", "PRKCM 2023-AFC1",
    "VERUS 2023-7", "VISIO 2023-2", "ACRA 2024-NQM1", "ADMT 2024-NQM5",
    "AOMT 2024-6", "BARC 2024-NQM2", "BRAVO 2024-NQM5",
    "COLT 2024-INV2", "CROSS 2024-H7", "DRMT 2024-1", "EFMT 2024-INV2",
    "GRADE 2024-INV1", "GSMBS 2024-NQM1", "HOMES 2024-AFC1",
    "JPMMT 2024-VIS1", "LMAT 2024-INV1", "MFRA 2024-NQM3",
    "MSRM 2024-NQM4", "NRZT 2024-NQM2", "NYMT 2024-INV1",
    "OBX 2024-NQM15", "PRKCM 2024-AFC1", "PRPM 2024-NQM1",
    "VERUS 2024-9", "ADMT 2025-NQM4", "BARC 2025-NQM3",
    "BRAVO 2025-NQM8", "CIM 2025-I1", "CROSS 2025-H7",
    "DRMT 2025-INV1", "EFMT 2025-INV5", "GCAT 2025-NQM3",
    "GSMBS 2025-DSC2", "HOMES 2025-AFC4", "JPMMT 2025-NQM4",
    "MFRA 2025-NQM5", "MSRM 2025-DSC2", "NLT 2025-INV1",
    "NRZT 2025-NQM6", "NYMT 2025-INV1", "OBX 2025-NQM21",
    "PRKCM 2025-AFC2", "PRPM 2025-NQM1", "VERUS 2025-11",
    "GSMBS 2026-NQM1", "JPMMT 2026-NQM1", "NYMT 2026-INV1",
    "OBX 2026-NQM1", "VERUS 2026-1",
]

deal_filter_df = pd.DataFrame({
    "bbg_deal_name": deal_name_filter,
    "filter_order": range(len(deal_name_filter)),
})

deal_name_list = ", ".join(f"'{deal_name}'" for deal_name in deal_name_filter)
deal_name_sql = f"""
    select distinct
        pool_id,
        bbg_deal_name
    from libremax.dbo.CRTStats
    where bbg_deal_name in ({deal_name_list})
    order by bbg_deal_name, pool_id
"""

deal_name_lookup = lmdb.executeSQLTableDF(db="libremax", sql_query=deal_name_sql, redshift=False)

def count_filter_data_rows(path: Path) -> int:
    with path.open("rb") as f:
        return sum(1 for _ in f) - 1

filtered_deals = deal_filter_df.merge(deal_name_lookup, on="bbg_deal_name", how="left")
missing_deal_names = filtered_deals.loc[filtered_deals["pool_id"].isna(), "bbg_deal_name"].tolist()
if missing_deal_names:
    raise ValueError("Deal names not found in CRTStats:\n" + "\n".join(missing_deal_names))

filtered_deals["file_path"] = filtered_deals["pool_id"].map(pool_file_lookup)
missing_files = filtered_deals.loc[filtered_deals["file_path"].isna(), ["pool_id", "bbg_deal_name"]]
if not missing_files.empty:
    warnings.warn(
        "Skipping deals with no monthly flat file:\n" + missing_files.to_string(index=False),
        stacklevel=2,
    )
    filtered_deals = filtered_deals.dropna(subset=["file_path"]).copy()

filtered_deals["data_rows"] = filtered_deals["file_path"].map(count_filter_data_rows)
filtered_deals = filtered_deals.sort_values("filter_order")

source_files = filtered_deals["file_path"].tolist()
output_pool_id = "NQM_DEAL_FILTER"
output_path = source_dir / f"{output_pool_id}_{asof_date}.txt"

print(f"Selected {len(filtered_deals):,} filtered deals with {filtered_deals['data_rows'].sum():,} total expected rows")
display(filtered_deals[["pool_id", "bbg_deal_name", "data_rows"]])

len(source_files), output_path

c:\qr\miniconda\envs\pyprod\Lib\site-packages\IPython\core\interactiveshell.py:3577: UserWarning: Skipping deals with no monthly flat file:
pool_id   bbg_deal_name
    ENO HOMES 2023-NQM1
    ENC MCMLT 2023-NQM1
  exec(code_obj, self.user_global_ns, self.user_ns)


Selected 99 filtered deals with 92,715 total expected rows


,pool_id,bbg_deal_name,data_rows
0,QC2,AOMT 2019-6,372
1,QVX,ARRW 2019-2,454
2,QC6,BRAVO 2019-2,1392
3,QCO,BRAVO 2019-2,857
4,QQS,FSMT 2018-3INV,293
...,...,...,...
96,HWI,GSMBS 2026-NQM1,1019
97,HWP,JPMMT 2026-NQM1,1162
98,HW3,NYMT 2026-INV1,1713
99,HXF,OBX 2026-NQM1,1599


(99,
 WindowsPath('N:/FlatFilesMonthly/NQMMonthly/NQM_DEAL_FILTER_20260501.txt'))

In [17]:
def count_lines(path: Path) -> int:
    with path.open("rb") as f:
        return sum(1 for _ in f)

missing = [path for path in source_files if not path.exists()]
if missing:
    raise FileNotFoundError("Missing source files:\n" + "\n".join(str(path) for path in missing))

expected_rows = 0
for path in source_files:
    data_rows = count_lines(path) - 1
    expected_rows += data_rows
    print(f"{path.name}: {data_rows:,} data rows")

print(f"Total expected rows: {expected_rows:,}")

QC2_20260501.txt: 372 data rows
QVX_20260501.txt: 454 data rows
QC6_20260501.txt: 1,392 data rows
QCO_20260501.txt: 857 data rows
QQS_20260501.txt: 293 data rows
QP6_20260501.txt: 3,338 data rows
Q74_20260501.txt: 418 data rows
QXG_20260501.txt: 445 data rows
QXU_20260501.txt: 405 data rows
Q43_20260501.txt: 1,901 data rows
Q91_20260501.txt: 341 data rows
B9R_20260501.txt: 356 data rows
Q9H_20260501.txt: 753 data rows
ENQ_20260501.txt: 699 data rows
BXX_20260501.txt: 578 data rows
BYG_20260501.txt: 530 data rows
BYQ_20260501.txt: 500 data rows
BX7_20260501.txt: 791 data rows
Q7V_20260501.txt: 699 data rows
BXV_20260501.txt: 557 data rows
BXK_20260501.txt: 706 data rows
BY0_20260501.txt: 806 data rows
B3I_20260501.txt: 864 data rows
B8V_20260501.txt: 820 data rows
B8P_20260501.txt: 776 data rows
EL9_20260501.txt: 1,339 data rows
B4Y_20260501.txt: 904 data rows
B5P_20260501.txt: 783 data rows
B8K_20260501.txt: 868 data rows
B3W_20260501.txt: 720 data rows
EKV_20260501.txt: 737 data rows


In [18]:
def combine_flatfiles(source_files: list[Path], output_path: Path) -> int:
    """Combine tab-delimited pool flat files with one shared header."""
    header = None
    data_rows = 0

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8", newline="") as out_file:
        for path in source_files:
            with path.open("r", encoding="utf-8", newline="") as in_file:
                current_header = in_file.readline()
                if not current_header:
                    raise ValueError(f"Empty source file: {path}")

                if header is None:
                    header = current_header
                    out_file.write(header)
                elif current_header != header:
                    raise ValueError(f"Header mismatch in {path}")

                for line in in_file:
                    out_file.write(line)
                    data_rows += 1

    return data_rows

rows_written = combine_flatfiles(source_files, output_path)
print(f"Wrote {rows_written:,} rows to {output_path}")

Wrote 92,715 rows to N:\FlatFilesMonthly\NQMMonthly\NQM_DEAL_FILTER_20260501.txt


In [ ]:
# Validate the combined file row count.
expected_rows = sum(count_lines(path) - 1 for path in source_files)
actual_rows = count_lines(output_path) - 1

print(f"Expected rows: {expected_rows:,}")
print(f"Actual rows:   {actual_rows:,}")
assert actual_rows == expected_rows

print("Combined flat file is ready:")
print(output_path)

Expected rows: 92,715
Actual rows:   92,715
Combined flat file is ready:
N:\FlatFilesMonthly\NQMMonthly\NQM_DEAL_FILTER_20260501.txt


: 

## Run LMSim2

Use `LMSim2/test_config_nqm_top10.json` in the LMSim repo to run this combined flat file. That config expects the combined file name to be `NQM_TOP10_20260401.txt` in `N:/FlatFilesMonthly/NQMMonthly/`.